## 데이터 수집 자동화

### 셀레니움

- 웹 브라우저를 자동으로 제어하는 파이썬 라이브러리(패키지)

- 기능
    - 브라우저 실행
    - 웹 페이지 로드(접속)
    - 입력창에 글자 입력(아이디, 패스워드 등)
    - 버튼 클릭
    - 페이지 이동
    - 화면에 표시된 텍스트 추출
    - 특정 이미지 저장 / 화면 스크린 캡처
    - 반복 자동화

- 정적인 HTML 은 Beautifulsoup4, request 패키지로 수집가능
- 동적인 웹화면은 Selenium이 강력함. 정적인 웹 페이지도 가능    

#### 웹브라우저 실행

In [1]:
# 셀레니움 패키지 로드
from selenium import webdriver
import pandas as pd

In [2]:
# 웹 브라우저 프로세서를 파이썬이 driver 변수로 관리
driver = webdriver.Chrome()

driver.get(f'http://www.naver.com')

In [3]:
# 웹 브라우저 종료
driver.quit()

#### 웹 페이지 스크래핑 실습

In [4]:
driver = webdriver.Chrome()

driver.get(f'http://127.0.0.1:5500/chapt04/products.html')

#### 요소 찾기

- 주요 요소 찾기 방식
    - **`By.ID`** : ID로 찾기
    - **`By.CLASS_NAME`** : class명으로 찾기
    - `By.TAG_NAEM` : 태크명으로 찾기
    - **`By.CSS_SELECTOR`** : CSS 선택자 방법으로 찾기
    - `By.XPATH` : Xpath로 찾기 

In [5]:
# Css 선택자로 요소찾는 함수용 패키지 내 클래스 로드
from selenium.webdriver.common.by import By

In [6]:
# 페이지 내 하나의 데이터만 추출
title = driver.find_element(By.ID, 'title')

title.text

'상품목록'

In [13]:
# 페이지 내 여러 요소를 동시에 추출
products = driver.find_elements(By.CLASS_NAME, 'product')

print(len(products))

3


In [14]:
products = driver.find_elements(By.CLASS_NAME, 'product')

for product in products:
    print(product.text)

키보드 30000
마우스 15000
모니터 250000


##### CSS 선택자로 요소 추출

In [19]:
products = driver.find_elements(By.CSS_SELECTOR,'.product')

for product in products:
    print(product)
    name = product.find_element(By.CSS_SELECTOR,'.name').text
    price = product.find_element(By.CSS_SELECTOR, '.price').text
    print(name,price)

<selenium.webdriver.remote.webelement.WebElement (session="6248872aea624b3e48e209929f4c6c8a", element="f.F514E6A90813361AF0BCF32473DADF32.d.B0F60200CA20E53179A710168A09DAFA.e.1337")>
키보드 30000
<selenium.webdriver.remote.webelement.WebElement (session="6248872aea624b3e48e209929f4c6c8a", element="f.F514E6A90813361AF0BCF32473DADF32.d.B0F60200CA20E53179A710168A09DAFA.e.1338")>
마우스 15000
<selenium.webdriver.remote.webelement.WebElement (session="6248872aea624b3e48e209929f4c6c8a", element="f.F514E6A90813361AF0BCF32473DADF32.d.B0F60200CA20E53179A710168A09DAFA.e.1339")>
모니터 250000


#### 입력창 입력

In [35]:
keyword_input = driver.find_element(By.ID, 'keyword')
keyword_input.send_keys('USB');

In [36]:
keyword_input.clear()

In [37]:
keyword_input.send_keys('테슬라')

In [38]:
# 입력태그에서 텍스트 가져오기
keyword_str = driver.find_element(By.ID,'keyword')

keyword_str.get_attribute('value')

'테슬라'

##### 버튼 클릭

In [40]:
button = driver.find_element(By.ID, 'search-btn')
button.click()

##### 명시적 대기

- 링크 클릭으로 다른 페이지 이동, 검색 후 결과 대기까지 시간이 소요
- `time.sleep(2)` 2초동안 대기 등의 파이썬 기본 함수가 제약사항이 많음
- 특정 조건이 만족할때까지 대기하는 로직 사용

In [41]:
# 명시적 대기를 위한 패키지 로드
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [43]:
# 여러 작업 수행 후

wait = WebDriverWait(driver, 10) #10초까지 대기

title = wait.until(
    EC.presence_of_element_located((By.ID, 'title'))
)

title.text

'상품목록'

##### 데이터 수집하기

In [48]:
products = driver.find_elements(By.CLASS_NAME,'product')

result = []

for product in products:
    name = product.find_element(By.CSS_SELECTOR, '.name').text
    price = product.find_element(By.CLASS_NAME, 'price').text

    result.append({ 'name':name, 'price':price})

In [49]:
result

[{'name': '키보드', 'price': '30000'},
 {'name': '마우스', 'price': '15000'},
 {'name': '모니터', 'price': '250000'}]

In [50]:
df = pd.DataFrame(result)
df

,name,price
0,키보드,30000
1,마우스,15000
2,모니터,250000


In [51]:
df.to_csv('../data/scrap_products.csv', encoding='utf-8')

##### 정리

- 웹 스크래핑시 필요요소
    - 웹드라이버로 웹사이트 오픈
    - 필요 요소 찾기
    - 입력태그 찾기
    - 버튼 요소 찾기
    - 입력하고 버튼 클릭해서 동작
    - 데이터 가져오기
    - 화면이 갱신되기까지 명시적 대기

#### 실습

##### 멜론 사이트 TOP00 스크래핑

In [91]:
# 페이지 전환없는 스크래핑용 패키지 로드
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [98]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://www.melon.com/chart/index.htm')

company_name = wait.until(
    EC.presence_of_element_located((By.ID, 'footer'))
)



#### 차트의 공 행 가져오기

In [104]:
# CSS 선택자 입력시 태그 + .css 동일한 기능
# songs = driver.find_elements(By.CSS_SELECTOR, '.lst50, .lst100')
songs = driver.find_elements(By.CSS_SELECTOR, 'tr.lst50, tr.lst100')

# len(songs)
for song in songs:
    # 노래 순위
    rank = song.find_element(By.CSS_SELECTOR, '.rank').text
    # 노래 제목
    title = song.find_element(By.CSS_SELECTOR, '.rank01 > span >a').text
    # 가수명
    singer = song.find_element(By.CSS_SELECTOR, '.rank02 a').text
    # 앨범명
    albumn = song.find_element(By.CSS_SELECTOR,'.rank03 a').text
    
    # 좋아요 개수
    like = song.find_element(By.CLASS_NAME, 'cnt').text

    # 단계 상승하락
    rank_ud = song.find_element(By.CSS_SELECTOR, '.rank_wrap').get_attribute('title')
    
    # 이미지 썸네일 주소 /생략
    img_url = song.find_element(By.CSS_SELECTOR, '.image_typeAll img').get_attribute('src')
    
    
    print(rank, rank_ud,title,singer,albumn,like)
    
    #print(rank,title,singer,albumn,like)

1 순위 동일 퇴사할게여 (Narr. 기안84) 소연 (SOYEON) 끝내주는 인생 39,517
2 순위 동일 LOVE ATTACK RESCENE (리센느) SCENEDROME 153,142
3 순위 동일 갑자기 아이오아이 (I.O.I) I.O.I 3rd MINI ALBUM [I.O.I : LOOP] 80,177
4 순위 동일 Deja Vu RESCENE (리센느) Dearest 58,937
5 순위 동일 Pretty Girl RESCENE (리센느) Pretty Girl - Special Single 66,158
6 순위 동일 REDRED CORTIS (코르티스) GREENGREEN 91,127
7 순위 동일 만찬가 태연 (TAEYEON) J-POP REMAKE Vol.1 49,656
8 순위 동일 BAD ATEEZ(에이티즈) GOLDEN HOUR : Part.5 33,783
9 순위 동일 BiiiG BIGBANG (빅뱅) BiiiG 56,954
10 순위 동일 LEMONADE aespa LEMONADE - The 2nd Album 59,587
11 순위 동일 Drowning WOODZ OO-LI 309,370
12 순위 동일 It′s Me 아일릿(ILLIT) MAMIHLAPINATAPAI 59,753
13 순위 동일 이 별로부터 아이유 이 별로부터 31,645
14 1단계 상승 Dear my crazy soulmate 아이유 이 별로부터 28,189
15 1단계 하락 Good Goodbye 화사 (HWASA) Good Goodbye 125,781
16 1단계 상승 사랑하게 될 거야 한로로 이상비행 149,812
17 1단계 하락 노스탈지아 BIG Naughty (서동현) 노스탈지아 32,575
18 순위 동일 RUDE! Hearts2Hearts (하츠투하츠) RUDE! 75,754
19 순위 동일 캐치 캐치 YENA (최예나) LOVE CATCHER 52,063
20 1단계 상승 어떻게 이별까지 사랑하겠어, 널 사랑하는 거지 AKMU (악뮤) 항해 496,

In [ ]:
songs = driver.find_elements(By.CSS_SELECTOR, 'tr.lst50, tr.lst100')

datas =[]

# len(songs)
for song in songs:
    # 노래 순위
    rank = song.find_element(By.CSS_SELECTOR, '.rank').text
    # 노래 제목
    title = song.find_element(By.CSS_SELECTOR, '.rank01 > span >a').text
    # 가수명
    singer = song.find_element(By.CSS_SELECTOR, '.rank02 a').text
    # 앨범명
    albumn = song.find_element(By.CSS_SELECTOR,'.rank03 a').text
    
    # 좋아요 개수
    like = song.find_element(By.CLASS_NAME, 'cnt').text

    # 단계상승하락
    rank_ud = song.find_element(By.CSS_SELECTOR, '.rank_wrap').get_attribute('title')

    datas.append({
        '순위' : rank,
        '순위변동' : rank_ud,
        '노래제목' : title,
        '가수' : singer,
        '앨범' : albumn,
        '좋아요' : like

    })

In [88]:
driver.quit()

In [105]:
df_melon100 = pd.DataFrame(datas)
df_melon100

,순위,순위변동,노래제목,가수,앨범,좋아요
0,1,1단계 하락,퇴사할게여 (Narr. 기안84),소연 (SOYEON),끝내주는 인생,"39,517"
1,2,1단계 하락,LOVE ATTACK,RESCENE (리센느),SCENEDROME,"153,142"
2,3,1단계 하락,갑자기,아이오아이 (I.O.I),I.O.I 3rd MINI ALBUM [I.O.I : LOOP],"80,177"
3,4,1단계 하락,Deja Vu,RESCENE (리센느),Dearest,"58,937"
4,5,1단계 하락,Pretty Girl,RESCENE (리센느),Pretty Girl - Special Single,"66,158"
...,...,...,...,...,...,...
95,96,1단계 하락,Less than a Lover,제니 (JENNIE),Less than a Lover,"27,099"
96,97,1단계 하락,New York,임영웅,IM HERO 10,"24,268"
97,98,1단계 하락,그댈 위한 멜로디,임영웅,IM HERO 2,"42,734"
98,99,1단계 하락,사랑은 봄비처럼...이별은 겨울비처럼...,임현정,All That Love,"76,461"


In [90]:
df_melon100.to_csv('../data/멜론Top100_261001.csv', encoding='utf-8')